In [51]:
import os
import librosa
import numpy as np
from transformers import BertTokenizer, BertModel
import torch
import torchaudio
from torchaudio.transforms import MelSpectrogram

In [52]:
import pandas as pd

In [53]:
df = pd.read_csv('../data/musiccaps-public.csv')

In [54]:
df.head()

,ytid,start_s,end_s,audioset_positive_labels,aspect_list,caption,author_id,is_balanced_subset,is_audioset_eval
0,-0Gj8-vB1q4,30,40,"/m/0140xf,/m/02cjck,/m/04rlf","['low quality', 'sustained strings melody', 's...",The low quality recording features a ballad so...,4,False,True
1,-0SdAVK79lg,30,40,"/m/0155w,/m/01lyv,/m/0342h,/m/042v_gx,/m/04rlf...","['guitar song', 'piano backing', 'simple percu...",This song features an electric guitar as the m...,0,False,False
2,-0vPFx-wRRI,30,40,"/m/025_jnm,/m/04rlf","['amateur recording', 'finger snipping', 'male...",a male voice is singing a melody with changing...,6,False,True
3,-0xzrMun0Rs,30,40,"/m/01g90h,/m/04rlf","['backing track', 'jazzy', 'digital drums', 'p...",This song contains digital drums playing a sim...,6,False,True
4,-1LrH01Ei1w,30,40,"/m/02p0sh1,/m/04rlf","['rubab instrument', 'repetitive melody on dif...",This song features a rubber instrument being p...,0,False,False


In [55]:
exist = []
for file in os.listdir('../data/music_data'):
    if file[:-4] in list(df['ytid']):
        exist.append(file[:-4])

In [56]:
filtered_df = df[df['ytid'].isin(exist)]

In [57]:
filtered_df

,ytid,start_s,end_s,audioset_positive_labels,aspect_list,caption,author_id,is_balanced_subset,is_audioset_eval
0,-0Gj8-vB1q4,30,40,"/m/0140xf,/m/02cjck,/m/04rlf","['low quality', 'sustained strings melody', 's...",The low quality recording features a ballad so...,4,False,True
1,-0SdAVK79lg,30,40,"/m/0155w,/m/01lyv,/m/0342h,/m/042v_gx,/m/04rlf...","['guitar song', 'piano backing', 'simple percu...",This song features an electric guitar as the m...,0,False,False
2,-0vPFx-wRRI,30,40,"/m/025_jnm,/m/04rlf","['amateur recording', 'finger snipping', 'male...",a male voice is singing a melody with changing...,6,False,True
3,-0xzrMun0Rs,30,40,"/m/01g90h,/m/04rlf","['backing track', 'jazzy', 'digital drums', 'p...",This song contains digital drums playing a sim...,6,False,True
4,-1LrH01Ei1w,30,40,"/m/02p0sh1,/m/04rlf","['rubab instrument', 'repetitive melody on dif...",This song features a rubber instrument being p...,0,False,False
...,...,...,...,...,...,...,...,...,...
995,9Kut4r8hswE,30,40,"/m/04rlf,/m/07gxw,/m/07lnk,/m/0m0jc","['low quality', 'electro', 'noisy traffic soun...",The low quality recording features an electro ...,4,False,True
996,9L6ePkWtZI4,30,40,"/m/04rlf,/m/0gg8l","['low quality', 'acoustic sitar chord progress...",The low quality recording features a cover of ...,4,False,True
997,9Lst8RagMYs,500,510,"/m/04rlf,/m/06rqw","['ska song', 'no voices', 'instrumental', 'bra...",This ska song features the main melody played ...,0,False,False
998,9M4IT3lOU10,30,40,"/m/04rlf,/m/07swgks","['low quality music', 'latin rhythm', 'male vo...",This clip features very low quality recorded m...,0,False,True


In [58]:
filtered_df = filtered_df[['ytid', 'caption']]

In [59]:
filtered_df.head()

,ytid,caption
0,-0Gj8-vB1q4,The low quality recording features a ballad so...
1,-0SdAVK79lg,This song features an electric guitar as the m...
2,-0vPFx-wRRI,a male voice is singing a melody with changing...
3,-0xzrMun0Rs,This song contains digital drums playing a sim...
4,-1LrH01Ei1w,This song features a rubber instrument being p...


In [60]:
filtered_df.to_csv('../data/filtered.csv')

In [61]:
# Set paths
DATASET_PATH = "../data/"
AUDIO_PATH = os.path.join(DATASET_PATH, "music_data/")
TEXT_FILE = os.path.join(DATASET_PATH, "filtered.csv")

In [62]:
# 1. Text Preprocessing
def preprocess_text(text_descriptions):
    """
    Tokenize and encode text descriptions using BERT.
    """
    tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
    model = BertModel.from_pretrained("bert-base-uncased")
    embeddings = []

    for text in text_descriptions:
        inputs = tokenizer(text, return_tensors="pt", truncation=True, padding="max_length", max_length=128)
        with torch.no_grad():
            outputs = model(**inputs)
        # Use the [CLS] token embedding
        cls_embedding = outputs.last_hidden_state[:, 0, :].squeeze(0).numpy()
        embeddings.append(cls_embedding)
    
    return np.array(embeddings)

In [63]:
# 2. Audio Preprocessing
def preprocess_audio(audio_path, target_sr=16000, n_mels=128):
    """
    Convert audio files to mel spectrograms.
    """
    audio_files = [f+'.wav' for f in audio_path]
    spectrograms = []

    for file in audio_files:
        file_path = os.path.join('../data/music_data/', file)
        try:
            # y, sr = librosa.load(file_path, sr=target_sr)

            # mel_spec = librosa.feature.melspectrogram(y, sr=sr, n_mels=n_mels)

            
            waveform, sample_rate = torchaudio.load(file_path)

            # Create MelSpectrogram transform
            transform = MelSpectrogram(sample_rate=sample_rate, n_mels=128)

            # Apply transform
            mel_spec = transform(waveform)
            
            log_mel_spec = librosa.power_to_db(mel_spec, ref=np.max)
            spectrograms.append(log_mel_spec)

        except Exception as e:
            print(f"Error processing {file}: {e}")
    
    return spectrograms #np.array(spectrograms)

In [64]:
# 3. Main Function
def preprocess_data(text_file):
    """
    Preprocess text and audio data.
    """
    # Load text descriptions
    import pandas as pd
    text_df = pd.read_csv(text_file)
    text_descriptions = text_df['caption'].tolist()

    # Preprocess text and audio
    print('Starting text Emnedding...')
    text_embeddings = preprocess_text(text_descriptions)
    print()
    print('Starting Audio Processing...')
    audio_features = preprocess_audio(text_df['ytid'])

    return text_embeddings, audio_features

In [65]:
# Preprocess the data
text_embeddings, audio_features = preprocess_data(TEXT_FILE)

# Save preprocessed data
np.save("text_embeddings.npy", text_embeddings)
# np.save("audio_features.npy", audio_features)

Starting text Emnedding...

Starting Audio Processing...


/Users/mohamedaminemrabet/miniforge3/envs/tfnew/lib/python3.9/site-packages/torchaudio/functional/functional.py:584: UserWarning: At least one mel filterbank has all zero values. The value for `n_mels` (128) may be set too high. Or, the value for `n_freqs` (201) may be set too low.
  warnings.warn(
/Users/mohamedaminemrabet/miniforge3/envs/tfnew/lib/python3.9/site-packages/torchaudio/functional/functional.py:584: UserWarning: At least one mel filterbank has all zero values. The value for `n_mels` (128) may be set too high. Or, the value for `n_freqs` (201) may be set too low.
  warnings.warn(


In [66]:
# fixed_size = (2, 128, 2401)

def pad_or_truncate(feature, size):
    # Create a zero-padded array of the desired size
    padded = np.zeros(size, dtype=feature.dtype)
    
    # Determine the minimum dimensions to copy
    min_dim1 = min(size[0], feature.shape[0])
    min_dim2 = min(size[1], feature.shape[1])
    min_dim3 = min(size[2], feature.shape[2])
    
    # Copy the data from the feature array
    padded[:min_dim1, :min_dim2, :min_dim3] = feature[:min_dim1, :min_dim2, :min_dim3]
    
    return padded

# # Apply padding/truncation
# audio_features_padded = [pad_or_truncate(feature, fixed_size) for feature in audio_features]

# # Convert to NumPy array and save
# audio_features_padded = np.array(audio_features_padded)
# np.save("audio_features_padded.npy", audio_features_padded)

In [67]:
text_em = np.load('text_embeddings.npy')
audio_em = np.load('audio_features_padded.npy')

In [68]:
audio_em.shape

(975, 2, 128, 2401)

# Model Architecture

In [69]:
import tensorflow as tf
from keras import Model, layers

class MusicGenerationModel(tf.keras.Model):
    def __init__(self, text_embedding_dim, audio_feature_dim, hidden_dim, seq_len):
        super(MusicGenerationModel, self).__init__()
        
        self.seq_len = seq_len
        
        self.text_to_hidden = layers.Dense(hidden_dim, activation='relu')
        
        self.lstm = layers.LSTM(
            hidden_dim, 
            return_sequences=True, 
            return_state=False, 
            recurrent_initializer='glorot_uniform'
        )
        
        self.decoder = tf.keras.Sequential([
            layers.Dense(hidden_dim, activation='relu'),
            layers.Dense(audio_feature_dim)
        ])
    
    def call(self, inputs):
        hidden_state = self.text_to_hidden(inputs)
        repeated_hidden = tf.repeat(tf.expand_dims(hidden_state, axis=1), self.seq_len, axis=1)
        lstm_out = self.lstm(repeated_hidden)
        audio_features = self.decoder(lstm_out)
        return audio_features


# Hyperparameters
text_embedding_dim = 768  # Example size for embeddings from a BERT model
audio_feature_dim = 2401  # Example size for spectrogram features
hidden_dim = 512
seq_len = 2  # Example length of output audio sequence

# Initialize the model
model = MusicGenerationModel(
    text_embedding_dim=text_embedding_dim,
    audio_feature_dim=audio_feature_dim,
    hidden_dim=hidden_dim,
    seq_len=seq_len
)

# Example input
batch_size = 16
text_embeddings = tf.random.normal((batch_size, text_embedding_dim))

# Forward pass
output = model(text_em)
print("Output shape:", output.shape)  # Expected: (batch_size, seq_len, audio_feature_dim)


Output shape: (975, 2, 2401)


In [70]:
model.summary()

Model: "music_generation_model_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_30 (Dense)                │ (975, 512)             │       393,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_10 (LSTM)                  │ (975, 2, 512)          │     2,099,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_10 (Sequential)      │ (975, 2, 2401)         │     1,494,369 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,987,297 (15.21 MB)

 Trainable params: 3,987,297 (15.21 MB)

 Non-trainable params: 0 (0.00 B)

In [71]:
# Reshape audio features to match (batch_size, seq_len, audio_feature_dim)
audio_em_resized = audio_em.reshape(audio_em.shape[0], 2, -1)  # Flatten the mel bins with the features
# audio_em_resized = audio_em.reshape(audio_em.shape[0], 2, 2401)  # (975, 2, 2401)

In [73]:
model.compile(optimizer='adam', loss='mse')

In [74]:
text_embeddings_batch = text_em[:16]  # Choose a batch size, e.g., 16
audio_em_batch = audio_em_resized[:16]  # Match batch size for audio features

# Train the model
model.fit(x=text_embeddings_batch, y=audio_em_batch, batch_size=16, epochs=5)

Epoch 1/5


ValueError: Dimensions must be equal, but are 307328 and 2401 for '{{node compile_loss/mse/sub}} = Sub[T=DT_FLOAT](data_1, music_generation_model_10_1/sequential_10_1/dense_32_1/BiasAdd)' with input shapes: [16,2,307328], [16,2,2401].

In [75]:
text_embedding_dim = 768  # BERT output size for [CLS] token
audio_feature_dim = 2401  # Output size for each audio feature
hidden_dim = 512
seq_len = 2  # Length of output audio sequence

# Input layers
text_input = tf.keras.Input(shape=(text_embedding_dim,))
audio_input = tf.keras.Input(shape=(seq_len, audio_feature_dim))

# Dense layer for text embedding processing
text_hidden = tf.keras.layers.Dense(hidden_dim, activation='relu')(text_input)

# Repeat text embeddings for sequence length
text_rep = tf.keras.layers.RepeatVector(seq_len)(text_hidden)

# LSTM for sequential modeling
lstm_out = tf.keras.layers.LSTM(
    hidden_dim, 
    return_sequences=True, 
    recurrent_initializer='glorot_uniform'
)(text_rep)

# Decoder layer to generate audio features
audio_output = tf.keras.layers.Dense(audio_feature_dim)(lstm_out)

# Model definition
model = tf.keras.Model(inputs=text_input, outputs=audio_output)

# Compile the model
model.compile(optimizer='adam', loss='mse')
model.summary()

Model: "functional_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_10 (InputLayer)     │ (None, 768)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_33 (Dense)                │ (None, 512)            │       393,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector (RepeatVector)    │ (None, 2, 512)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_11 (LSTM)                  │ (None, 2, 512)         │     2,099,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_34 (Dense)                │ (None, 2, 2401)        │     1,231,713 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,724,641 (14.21 MB)

 Trainable params: 3,724,641 (14.21 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
batch_size = 16
audio_features_padded_resized = audio_features_padded.reshape(audio_features_padded.shape[0], 2, -1)

# Batch data generator
def batch_data(text_embeddings, audio_em, batch_size):
    for i in range(0, len(text_embeddings), batch_size):
        yield text_embeddings[i:i+batch_size], audio_em[i:i+batch_size]

train_dataset = tf.data.Dataset.from_generator(
    lambda: batch_data(text_embeddings, audio_features_padded_resized, batch_size),
    output_signature=(
        tf.TensorSpec(shape=(None, text_embeddings.shape[1]), dtype=tf.float32),
        tf.TensorSpec(shape=(None, audio_features_padded_resized.shape[1], audio_features_padded_resized.shape[2]), dtype=tf.float32)
    )
)